In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(r"C:\India_Vs_West_Indies_T20i_Challenge")
CLEAN_FILE = PROJECT_DIR / "Data" / "Clean" / "t20i_ball_by_ball_clean.csv"

df = pd.read_csv(CLEAN_FILE)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

display(df.head())

Dataset loaded successfully!
Rows: 959
Columns: 24


,Venue,MatchDay,Innings,Batting Team,Bowling Team,Striker,Non-Striker,Bowler,Ball Number,Actual Ball Number,...,Wide,No Ball,Leg Bye,Bye,Wicket,4's,6's,Run_QC_Flag,Extras_QC_Flag,Ball_Number_Label
0,Ekana Cricket Stadium B Ground,1,1,West Indies Cricket Team,India National Cricket team,Shai Hope,Kamil Pooran,Arshdeep Singh,0.1,0.1,...,0,0,0,0,0,0,0,False,False,0.1
1,Ekana Cricket Stadium B Ground,1,1,West Indies Cricket Team,India National Cricket team,Shai Hope,Kamil Pooran,Arshdeep Singh,0.2,0.2,...,0,0,0,0,0,0,0,False,False,0.2
2,Ekana Cricket Stadium B Ground,1,1,West Indies Cricket Team,India National Cricket team,Shai Hope,Kamil Pooran,Arshdeep Singh,0.3,0.3,...,0,0,0,0,0,0,0,False,False,0.3
3,Ekana Cricket Stadium B Ground,1,1,West Indies Cricket Team,India National Cricket team,Shai Hope,Kamil Pooran,Arshdeep Singh,0.4,0.4,...,0,0,0,0,0,1,0,False,False,0.4
4,Ekana Cricket Stadium B Ground,1,1,West Indies Cricket Team,India National Cricket team,Shai Hope,Kamil Pooran,Arshdeep Singh,0.5,0.5,...,0,0,0,0,0,0,0,False,False,0.5


In [2]:
bowling_columns = [
    "Bowler",
    "Bowling Team",
    "Batting Team",
    "MatchDay",
    "Innings",
    "Ball Number",
    "Actual Ball Number",
    "Team Runs",
    "Runs Off The Bat",
    "Extra",
    "Wide",
    "No Ball",
    "Wicket",
    "4's",
    "6's",
]

print("Missing values in bowling columns:")
display(df[bowling_columns].isnull().sum())

print("\nUnique bowlers:", df["Bowler"].nunique())
print("Total delivery records:", len(df))
print("Total wicket events:", df["Wicket"].sum())

Missing values in bowling columns:


Bowler                0
Bowling Team          0
Batting Team          0
MatchDay              0
Innings               0
Ball Number           0
Actual Ball Number    0
Team Runs             0
Runs Off The Bat      0
Extra                 0
Wide                  0
No Ball               0
Wicket                0
4's                   0
6's                   0
dtype: int64


Unique bowlers: 17
Total delivery records: 959
Total wicket events: 50


In [3]:
# Check bowler names and their delivery counts
bowler_delivery_counts = (
    df.groupby("Bowler")
    .size()
    .reset_index(name="Delivery_Records")
    .sort_values("Delivery_Records", ascending=False)
)

display(bowler_delivery_counts)

print("Unique bowlers:", df["Bowler"].nunique())
print("Total wicket events:", df["Wicket"].sum())

,Bowler,Delivery_Records
1,Arshdeep Singh,105
12,Roston Chase,96
7,Mayank Yadav,95
15,Shamar Springer,83
2,Axar Patel,82
14,Shamar Joseph,74
8,Naman Dhir,63
11,Romario Shepherd,57
4,Keemo Paul,54
10,Prince Yadav,51


Unique bowlers: 17
Total wicket events: 50


In [4]:
# Treat missing extras as zero for calculations
for col in ["Wide", "No Ball", "Leg Bye", "Bye"]:
    df[col] = df[col].fillna(0)

# A legal delivery is one that is neither a wide nor a no-ball
df["Legal_Ball"] = (
    (df["Wide"] == 0) &
    (df["No Ball"] == 0)
).astype(int)

# Runs charged to the bowler:
# batter runs + wides + no-ball extras
df["Bowler_Runs_Conceded"] = (
    df["Runs Off The Bat"] +
    df["Wide"] +
    df["No Ball"]
)

# Summarize each bowler's workload and runs conceded
bowling_summary = (
    df.groupby(["Bowler", "Bowling Team"], as_index=False)
    .agg(
        Deliveries=("Bowler", "size"),
        Legal_Balls=("Legal_Ball", "sum"),
        Runs_Conceded=("Bowler_Runs_Conceded", "sum"),
        Wicket_Events=("Wicket", "sum"),
    )
)

# Convert legal balls to cricket overs
bowling_summary["Overs"] = (
    bowling_summary["Legal_Balls"] // 6
).astype(int).astype(str) + "." + (
    bowling_summary["Legal_Balls"] % 6
).astype(str)

# Economy rate uses legal balls, not decimal overs
bowling_summary["Economy"] = (
    bowling_summary["Runs_Conceded"] * 6 /
    bowling_summary["Legal_Balls"].replace(0, np.nan)
).round(2)

display(
    bowling_summary.sort_values(
        "Runs_Conceded",
        ascending=False
    )
)

,Bowler,Bowling Team,Deliveries,Legal_Balls,Runs_Conceded,Wicket_Events,Overs,Economy
1,Arshdeep Singh,India National Cricket team,105,96,175,5,16.0,10.94
2,Axar Patel,India National Cricket team,82,81,167,4,13.3,12.37
15,Shamar Springer,West Indies Cricket Team,83,78,166,6,13.0,12.77
14,Shamar Joseph,West Indies Cricket Team,74,69,163,2,11.3,14.17
7,Mayank Yadav,India National Cricket team,95,91,162,5,15.1,10.68
12,Roston Chase,West Indies Cricket Team,96,94,142,6,15.4,9.06
8,Naman Dhir,India National Cricket team,63,60,114,5,10.0,11.40
11,Romario Shepherd,West Indies Cricket Team,57,52,103,1,8.4,11.88
5,Kuldeep Yadav,India National Cricket team,49,48,86,3,8.0,10.75
0,Akeal Hosein,West Indies Cricket Team,44,42,83,1,7.0,11.86


In [5]:
# Check that legal balls equal completed overs plus remaining balls
bowling_summary["Legal_Balls_Check"] = (
    (bowling_summary["Legal_Balls"] // 6) * 6
    + (bowling_summary["Legal_Balls"] % 6)
)

# Independently verify economy rate
bowling_summary["Economy_Check"] = (
    bowling_summary["Runs_Conceded"] * 6
    / bowling_summary["Legal_Balls"].replace(0, np.nan)
).round(2)

print(
    "Legal-ball mismatches:",
    (
        bowling_summary["Legal_Balls"]
        != bowling_summary["Legal_Balls_Check"]
    ).sum()
)

print(
    "Economy mismatches:",
    (
        bowling_summary["Economy"]
        != bowling_summary["Economy_Check"]
    ).sum()
)

display(
    bowling_summary[
        [
            "Bowler",
            "Legal_Balls",
            "Overs",
            "Runs_Conceded",
            "Economy",
            "Wicket_Events",
        ]
    ].sort_values("Runs_Conceded", ascending=False)
)

Legal-ball mismatches: 0
Economy mismatches: 0


,Bowler,Legal_Balls,Overs,Runs_Conceded,Economy,Wicket_Events
1,Arshdeep Singh,96,16.0,175,10.94,5
2,Axar Patel,81,13.3,167,12.37,4
15,Shamar Springer,78,13.0,166,12.77,6
14,Shamar Joseph,69,11.3,163,14.17,2
7,Mayank Yadav,91,15.1,162,10.68,5
12,Roston Chase,94,15.4,142,9.06,6
8,Naman Dhir,60,10.0,114,11.40,5
11,Romario Shepherd,52,8.4,103,11.88,1
5,Kuldeep Yadav,48,8.0,86,10.75,3
0,Akeal Hosein,42,7.0,83,11.86,1


In [7]:
match_bowling_summary = (
    df.groupby(
        ["MatchDay", "Bowler", "Bowling Team"],
        as_index=False
    )
    .agg(
        Deliveries=("Bowler", "size"),
        Legal_Balls=("Legal_Ball", "sum"),
        Runs_Conceded=("Bowler_Runs_Conceded", "sum"),
        Wicket_Events=("Wicket", "sum"),
    )
)

# Display overs as cricket notation, not decimal arithmetic
match_bowling_summary["Overs"] = (
    (match_bowling_summary["Legal_Balls"] // 6).astype(int).astype(str)
    + "."
    + (match_bowling_summary["Legal_Balls"] % 6).astype(int).astype(str)
)

match_bowling_summary["Economy"] = (
    match_bowling_summary["Runs_Conceded"] * 6
    / match_bowling_summary["Legal_Balls"].replace(0, np.nan)
).round(2)

display(
    match_bowling_summary.sort_values(
        ["MatchDay", "Bowling Team", "Runs_Conceded"],
        ascending=[True, True, False]
    )
)

,MatchDay,Bowler,Bowling Team,Deliveries,Legal_Balls,Runs_Conceded,Wicket_Events,Overs,Economy
5,1,Mayank Yadav,India National Cricket team,25,25,40,2,4.1,9.60
1,1,Arshdeep Singh,India National Cricket team,25,24,34,1,4.0,8.50
3,1,Kuldeep Yadav,India National Cricket team,24,24,33,2,4.0,8.25
6,1,Naman Dhir,India National Cricket team,24,24,31,3,4.0,7.75
2,1,Axar Patel,India National Cricket team,18,18,26,2,3.0,8.67
4,1,Matthew Forde,West Indies Cricket Team,12,12,39,0,2.0,19.50
0,1,Akeal Hosein,West Indies Cricket Team,25,24,37,1,4.0,9.25
8,1,Roston Chase,West Indies Cricket Team,22,22,37,1,3.4,10.09
7,1,Romario Shepherd,West Indies Cricket Team,12,12,24,0,2.0,12.00
9,1,Shamar Joseph,West Indies Cricket Team,12,12,23,0,2.0,11.50


In [9]:
# Treat missing wide/no-ball values as zero
df["Wide"] = df["Wide"].fillna(0)
df["No Ball"] = df["No Ball"].fillna(0)

# A legal delivery is neither a wide nor a no-ball
df["Legal_Ball"] = (
    (df["Wide"] == 0) &
    (df["No Ball"] == 0)
).astype(int)

print("Legal_Ball column created successfully!")
print(df["Legal_Ball"].value_counts())

Legal_Ball column created successfully!
Legal_Ball
1    911
0     48
Name: count, dtype: int64


In [12]:
mayank_match1 = df.loc[
    (df["MatchDay"] == 1)
    & (df["Bowler"] == "Mayank Yadav"),
    [
        "MatchDay",
        "Innings",
        "Bowler",
        "Ball Number",
        "Actual Ball Number",
        "Wide",
        "No Ball",
        "Runs Off The Bat",
        "Extra",
        "Wicket",
        "Legal_Ball",
    ],
].copy()

display(mayank_match1)

print("Delivery records:", len(mayank_match1))
print("Legal balls:", mayank_match1["Legal_Ball"].sum())

,MatchDay,Innings,Bowler,Ball Number,Actual Ball Number,Wide,No Ball,Runs Off The Bat,Extra,Wicket,Legal_Ball
6,1,1,Mayank Yadav,1.1,1.1,0,0,4,0,0,1
7,1,1,Mayank Yadav,1.2,1.2,0,0,4,0,0,1
8,1,1,Mayank Yadav,1.3,1.3,0,0,0,0,0,1
9,1,1,Mayank Yadav,1.4,1.4,0,0,0,0,0,1
10,1,1,Mayank Yadav,1.5,1.5,0,0,4,0,0,1
11,1,1,Mayank Yadav,1.6,1.6,0,0,4,0,0,1
43,1,1,Mayank Yadav,7.1,7.1,0,0,0,0,0,1
44,1,1,Mayank Yadav,7.2,7.2,0,0,1,0,0,1
45,1,1,Mayank Yadav,7.3,7.3,0,0,1,0,0,1
46,1,1,Mayank Yadav,7.4,7.4,0,0,2,0,0,1


Delivery records: 25
Legal balls: 25


In [14]:
mayank_match1_duplicates = df.loc[
    (df["MatchDay"] == 1)
    & (df["Bowler"] == "Mayank Yadav")
].copy()

duplicate_columns = [
    "MatchDay",
    "Innings",
    "Bowler",
    "Ball Number",
    "Striker",
    "Non-Striker",
    "Runs Off The Bat",
    "Extra",
    "Wicket",
]

mayank_match1_duplicates = mayank_match1_duplicates[
    mayank_match1_duplicates.duplicated(
        subset=duplicate_columns,
        keep=False
    )
]

print("Potential duplicate records:", len(mayank_match1_duplicates))

display(
    mayank_match1_duplicates[
        duplicate_columns
    ]
)

Potential duplicate records: 0


,MatchDay,Innings,Bowler,Ball Number,Striker,Non-Striker,Runs Off The Bat,Extra,Wicket


In [16]:
print(df.columns.tolist())

['Venue', 'MatchDay', 'Innings', 'Batting Team', 'Bowling Team', 'Striker', 'Non-Striker', 'Bowler', 'Ball Number', 'Actual Ball Number', 'Overs', 'Team Runs', 'Runs Off The Bat', 'Extra', 'Wide', 'No Ball', 'Leg Bye', 'Bye', 'Wicket', "4's", "6's", 'Run_QC_Flag', 'Extras_QC_Flag', 'Ball_Number_Label', 'Legal_Ball', 'Bowler_Runs_Conceded']


In [17]:
mayank_check = df.loc[
    (df["MatchDay"] == 1)
    & (df["Innings"] == 1)
    & (df["Bowler"] == "Mayank Yadav"),
    [
        "Ball Number",
        "Striker",
        "Non-Striker",
        "Runs Off The Bat",
        "Wide",
        "No Ball",
        "Extra",
        "Wicket",
        "Legal_Ball",
    ],
].copy()

display(mayank_check.to_string(index=False))

print("Total records:", len(mayank_check))
print("Legal balls:", mayank_check["Legal_Ball"].sum())

' Ball Number             Striker         Non-Striker  Runs Off The Bat  Wide  No Ball  Extra  Wicket  Legal_Ball\n         1.1           Shai Hope        Kamil Pooran                 4     0        0      0       0           1\n         1.2           Shai Hope        Kamil Pooran                 4     0        0      0       0           1\n         1.3           Shai Hope        Kamil Pooran                 0     0        0      0       0           1\n         1.4           Shai Hope        Kamil Pooran                 0     0        0      0       0           1\n         1.5           Shai Hope        Kamil Pooran                 4     0        0      0       0           1\n         1.6           Shai Hope        Kamil Pooran                 4     0        0      0       0           1\n         7.1           Shai Hope Sherfane Rutherford                 0     0        0      0       0           1\n         7.2           Shai Hope Sherfane Rutherford                 1     0        0  

Total records: 25
Legal balls: 25


In [18]:
match1_bowling_check = (
    df.loc[
        (df["MatchDay"] == 1) &
        (df["Innings"] == 1)
    ]
    .groupby("Bowler")
    .agg(
        Delivery_Records=("Bowler", "size"),
        Legal_Balls=("Legal_Ball", "sum"),
        Runs_Off_Bat=("Runs Off The Bat", "sum"),
        Wickets=("Wicket", "sum")
    )
    .reset_index()
)

match1_bowling_check["Overs"] = (
    (match1_bowling_check["Legal_Balls"] // 6).astype(int).astype(str)
    + "."
    + (match1_bowling_check["Legal_Balls"] % 6).astype(int).astype(str)
)

display(
    match1_bowling_check.sort_values(
        "Legal_Balls", ascending=False
    ).to_string(index=False)
)

'        Bowler  Delivery_Records  Legal_Balls  Runs_Off_Bat  Wickets Overs\n  Mayank Yadav                25           25            40        2   4.1\nArshdeep Singh                25           24            33        1   4.0\n Kuldeep Yadav                24           24            33        2   4.0\n    Naman Dhir                24           24            31        3   4.0\n    Axar Patel                18           18            26        2   3.0'

In [19]:
mayank_over_check = (
    df.loc[
        (df["MatchDay"] == 1)
        & (df["Innings"] == 1)
        & (df["Bowler"] == "Mayank Yadav")
    ]
    .groupby("Ball Number", as_index=False)
    .agg(
        Delivery_Records=("Bowler", "size"),
        Legal_Balls=("Legal_Ball", "sum"),
        Runs=("Runs Off The Bat", "sum"),
        Extras=("Extra", "sum"),
        Wickets=("Wicket", "sum"),
    )
)

display(mayank_over_check.to_string(index=False))

' Ball Number  Delivery_Records  Legal_Balls  Runs  Extras  Wickets\n         1.1                 1            1     4       0        0\n         1.2                 1            1     4       0        0\n         1.3                 1            1     0       0        0\n         1.4                 1            1     0       0        0\n         1.5                 1            1     4       0        0\n         1.6                 1            1     4       0        0\n         7.1                 1            1     0       0        0\n         7.2                 1            1     1       0        0\n         7.3                 1            1     1       0        0\n         7.4                 1            1     2       0        0\n         7.5                 1            1     0       0        0\n         7.6                 1            1     0       4        0\n        16.1                 1            1     1       0        0\n        16.2                 1            1    

In [23]:
from pathlib import Path

RAW_DIR = Path(r"C:\India_Vs_West_Indies_T20i_Challenge\Data\Raw")

print("Does Raw folder exist?", RAW_DIR.exists())

if RAW_DIR.exists():
    for file in RAW_DIR.iterdir():
        print(file.name)

Does Raw folder exist? True
India_Vs_WestIndies_t20i_ball_by_ball_data - Sheet1 (1).csv


In [25]:
RAW_FILE = RAW_DIR / "India_Vs_WestIndies_t20i_ball_by_ball_data - Sheet1 (1).csv"

raw_df = pd.read_csv(RAW_FILE)

raw_mayank = raw_df.loc[
    (raw_df["MatchDay"] == 1)
    & (raw_df["Innings"] == 1)
    & (raw_df["Bowler"].astype(str).str.strip() == "Mayank Yadav"),
    ["Ball Number", "Bowler", "Striker", "Runs Off The Bat"]
]

display(raw_mayank.to_string(index=False))

print("Raw Mayank records:", len(raw_mayank))

' Ball Number       Bowler             Striker  Runs Off The Bat\n         1.1 Mayank Yadav           Shai Hope                 4\n         1.2 Mayank Yadav           Shai Hope                 4\n         1.3 Mayank Yadav           Shai Hope                 0\n         1.4 Mayank Yadav           Shai Hope                 0\n         1.5 Mayank Yadav           Shai Hope                 4\n         1.6 Mayank Yadav           Shai Hope                 4\n         7.1 Mayank Yadav           Shai Hope                 0\n         7.2 Mayank Yadav           Shai Hope                 1\n         7.3 Mayank Yadav Sherfane Rutherford                 1\n         7.4 Mayank Yadav           Shai Hope                 2\n         7.5 Mayank Yadav           Shai Hope                 0\n         7.6 Mayank Yadav           Shai Hope                 0\n        16.1 Mayank Yadav Sherfane Rutherford                 1\n        16.2 Mayank Yadav        Akeal Hosein                 1\n        16.3 Mayank Yada

Raw Mayank records: 25


In [26]:
raw_match1 = raw_df.loc[
    (raw_df["MatchDay"] == 1)
    & (raw_df["Innings"] == 1)
]

raw_bowling_check = (
    raw_match1.groupby("Bowler")
    .agg(
        Delivery_Records=("Bowler", "size"),
        Runs_Off_Bat=("Runs Off The Bat", "sum")
    )
    .reset_index()
    .sort_values("Delivery_Records", ascending=False)
)

display(raw_bowling_check.to_string(index=False))

'        Bowler  Delivery_Records  Runs_Off_Bat\nArshdeep Singh                25            33\n  Mayank Yadav                25            40\n Kuldeep Yadav                24            33\n    Naman Dhir                24            31\n    Axar Patel                18            26'

In [27]:
# Validate bowling data using the existing cleaned dataset

# 1. Check missing values in important bowling columns
bowling_columns = [
    "MatchDay",
    "Innings",
    "Bowling Team",
    "Bowler",
    "Runs Off The Bat",
    "Extra",
    "Wide",
    "No Ball",
    "Wicket",
    "Legal_Ball",
    "Bowler_Runs_Conceded",
]

print("Missing values in bowling columns:")
display(df[bowling_columns].isnull().sum().to_frame("Missing_Values"))

# 2. Check whether legal-ball flags are correct
expected_legal_ball = (
    (df["Wide"].fillna(0) == 0)
    & (df["No Ball"].fillna(0) == 0)
).astype(int)

print(
    "\nIncorrect legal-ball flags:",
    (df["Legal_Ball"] != expected_legal_ball).sum()
)

# 3. Check whether bowler runs conceded are calculated correctly
expected_bowler_runs = (
    df["Runs Off The Bat"].fillna(0)
    + df["Wide"].fillna(0)
    + df["No Ball"].fillna(0)
)

print(
    "Incorrect bowler-runs calculations:",
    (df["Bowler_Runs_Conceded"] != expected_bowler_runs).sum()
)

# 4. Check legal-ball totals by innings
innings_validation = (
    df.groupby(["MatchDay", "Innings"], as_index=False)
    .agg(
        Delivery_Records=("Bowler", "size"),
        Legal_Balls=("Legal_Ball", "sum"),
        Runs_Off_Bat=("Runs Off The Bat", "sum"),
        Extras=("Extra", "sum"),
        Wicket_Events=("Wicket", "sum"),
    )
)

innings_validation["Total_Team_Runs"] = (
    innings_validation["Runs_Off_Bat"] + innings_validation["Extras"]
)

display(innings_validation.to_string(index=False))

Missing values in bowling columns:


,Missing_Values
MatchDay,0
Innings,0
Bowling Team,0
Bowler,0
Runs Off The Bat,0
Extra,0
Wide,0
No Ball,0
Wicket,0
Legal_Ball,0



Incorrect legal-ball flags: 0
Incorrect bowler-runs calculations: 0


' MatchDay  Innings  Delivery_Records  Legal_Balls  Runs_Off_Bat  Extras  Wicket_Events  Total_Team_Runs\n        1        1               116          115           163       8             10              171\n        1        2                89           88           171       1              2              172\n        2        1               128          120           240       9              5              249\n        2        2               115          111           248       4              4              252\n        3        1               124          120           213      10             10              223\n        3        2               127          120           199      10              7              209\n        4        1               130          120           229      18              3              247\n        4        2               130          117           199      24              9              223'

In [28]:
# Build final bowling leaderboard

final_bowling_leaderboard = (
    df.groupby(["Bowler", "Bowling Team"], as_index=False)
    .agg(
        Innings_Bowled=("MatchDay", lambda x: df.loc[x.index, ["MatchDay", "Innings"]].drop_duplicates().shape[0]),
        Legal_Balls=("Legal_Ball", "sum"),
        Runs_Conceded=("Bowler_Runs_Conceded", "sum"),
        Wicket_Events=("Wicket", "sum"),
    )
)

# Convert legal balls to cricket overs notation
final_bowling_leaderboard["Overs"] = (
    (final_bowling_leaderboard["Legal_Balls"] // 6).astype(int).astype(str)
    + "."
    + (final_bowling_leaderboard["Legal_Balls"] % 6).astype(int).astype(str)
)

# Economy rate = runs conceded per six legal balls
final_bowling_leaderboard["Economy"] = (
    final_bowling_leaderboard["Runs_Conceded"] * 6
    / final_bowling_leaderboard["Legal_Balls"].replace(0, np.nan)
).round(2)

# Sort by wicket events, then economy
final_bowling_leaderboard = final_bowling_leaderboard.sort_values(
    ["Wicket_Events", "Economy"],
    ascending=[False, True]
).reset_index(drop=True)

display(final_bowling_leaderboard)

,Bowler,Bowling Team,Innings_Bowled,Legal_Balls,Runs_Conceded,Wicket_Events,Overs,Economy
0,Roston Chase,West Indies Cricket Team,4,94,142,6,15.4,9.06
1,Keemo Paul,West Indies Cricket Team,2,48,79,6,8.0,9.88
2,Shamar Springer,West Indies Cricket Team,4,78,166,6,13.0,12.77
3,Mayank Yadav,India National Cricket team,4,91,162,5,15.1,10.68
4,Arshdeep Singh,India National Cricket team,4,96,175,5,16.0,10.94
5,Naman Dhir,India National Cricket team,4,60,114,5,10.0,11.40
6,Gudakesh Motie,West Indies Cricket Team,3,48,71,4,8.0,8.88
7,Axar Patel,India National Cricket team,4,81,167,4,13.3,12.37
8,Kuldeep Yadav,India National Cricket team,2,48,86,3,8.0,10.75
9,Shamar Joseph,West Indies Cricket Team,4,69,163,2,11.3,14.17


In [29]:
print(final_bowling_leaderboard.columns.tolist())

display(
    final_bowling_leaderboard[
        [
            "Bowler",
            "Bowling Team",
            "Innings_Bowled",
            "Legal_Balls",
            "Runs_Conceded",
            "Wicket_Events",
            "Overs",
            "Economy",
        ]
    ].head(5).to_string(index=False)
)

['Bowler', 'Bowling Team', 'Innings_Bowled', 'Legal_Balls', 'Runs_Conceded', 'Wicket_Events', 'Overs', 'Economy']


'         Bowler                Bowling Team  Innings_Bowled  Legal_Balls  Runs_Conceded  Wicket_Events Overs  Economy\n   Roston Chase    West Indies Cricket Team               4           94            142              6  15.4     9.06\n     Keemo Paul    West Indies Cricket Team               2           48             79              6   8.0     9.88\nShamar Springer    West Indies Cricket Team               4           78            166              6  13.0    12.77\n   Mayank Yadav India National Cricket team               4           91            162              5  15.1    10.68\n Arshdeep Singh India National Cricket team               4           96            175              5  16.0    10.94'

In [30]:
print("Total legal balls:", final_bowling_leaderboard["Legal_Balls"].sum())
print("Total runs conceded:", final_bowling_leaderboard["Runs_Conceded"].sum())
print("Total wicket events:", final_bowling_leaderboard["Wicket_Events"].sum())

Total legal balls: 911
Total runs conceded: 1718
Total wicket events: 50


In [31]:
# Legal balls by innings
innings_balls = (
    df.groupby(["MatchDay", "Innings"])["Legal_Ball"]
    .sum()
    .sum()
)

# Legal balls by bowler
bowler_balls = final_bowling_leaderboard["Legal_Balls"].sum()

print("Legal balls from delivery data:", innings_balls)
print("Legal balls in bowling leaderboard:", bowler_balls)
print("Difference:", innings_balls - bowler_balls)

# Check whether any rows have missing bowler names
print("\nMissing bowler names:", df["Bowler"].isna().sum())

# Check whether any rows were excluded from the bowler grouping
print("Total delivery records:", len(df))
print(
    "Records included in bowling summary:",
    final_bowling_leaderboard["Legal_Balls"].sum()
)

Legal balls from delivery data: 911
Legal balls in bowling leaderboard: 911
Difference: 0

Missing bowler names: 0
Total delivery records: 959
Records included in bowling summary: 911


In [32]:
PROCESSED_DIR = PROJECT_DIR / "Data" / "Processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = PROCESSED_DIR / "final_bowling_leaderboard.csv"

final_bowling_leaderboard.to_csv(OUTPUT_FILE, index=False)

print("Bowling leaderboard saved successfully.")
print("File location:", OUTPUT_FILE)
print("Rows:", len(final_bowling_leaderboard))
print("Columns:", len(final_bowling_leaderboard.columns))

Bowling leaderboard saved successfully.
File location: C:\India_Vs_West_Indies_T20i_Challenge\Data\Processed\final_bowling_leaderboard.csv
Rows: 17
Columns: 8
